# Phần 2 – Biểu đồ so sánh chỉ số và xu hướng giá
Nhóm 5 PTDL · Trực quan hóa & Phân tích thống kê

Phần này **mô tả**, chưa kết luận: nhìn 8 ngân hàng qua 6 chỉ số, giá cổ phiếu và sự kiện để thấy bức tranh chung, làm nền cho câu 1, 2, 3.

Mọi biểu đồ vẽ bằng hàm trong `src/viz.py` (Hân dùng lại cho dashboard). Notebook này chỉ **gọi hàm, xem hình và lưu PNG** vào `outputs/figures/`.

| Mục | Nội dung trong kế hoạch | Hàm |
|---|---|---|
| 1 | So sánh 6 chỉ số giữa 8 ngân hàng: line theo quý, bar trung bình năm, boxplot | `ve_duong_chi_so`, `ve_cot_trung_binh_nam`, `ve_hop_chi_so`, `ve_tong_quan_chi_so` |
| 2 | Xu hướng giá: giá chuẩn hóa = 100, biên độ giá theo quý | `ve_gia_chuan_hoa`, `ve_bien_do_gia` |
| 3 | Sự kiện: số sự kiện theo ngân hàng/loại, đánh dấu trên đường giá | `ve_dem_su_kien`, `ve_gia_su_kien` |

In [19]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import config as cf
from src.data_loader import doc_tong_hop, doc_gia_ngay, doc_su_kien_chi_tiet
from src.viz import (ve_duong_chi_so, ve_cot_trung_binh_nam, ve_hop_chi_so, ve_tong_quan_chi_so,
                     ve_gia_chuan_hoa, ve_bien_do_gia, ve_dem_su_kien, ve_gia_su_kien, them_gia_chuan_hoa)

b4 = doc_tong_hop()            # 160 dòng: 8 NH × 20 quý
gia = doc_gia_ngay()           # giá ngày 04/01/2021 → 31/12/2025
su_kien = doc_su_kien_chi_tiet()  # 121 sự kiện
print(b4.shape, gia.shape, su_kien.shape)

(160, 60) (9976, 11) (121, 15)


**Lưu hình PNG:** đặt `LUU_HINH = True` để xuất PNG cho báo cáo. Cần Google Chrome; nếu báo lỗi Chrome thì chạy `plotly_get_chrome` trong Terminal một lần, hoặc tạm để `False` – biểu đồ vẫn hiện trong notebook.

In [20]:
LUU_HINH = True

def hien(fig, ten=None, show=True):
    """Hiện biểu đồ và (nếu LUU_HINH) lưu PNG vào outputs/figures/<ten>.png"""
    global LUU_HINH
    if show:
        fig.show()
    if LUU_HINH and ten:
        try:
            cf.luu_hinh(fig, ten)
        except Exception as e:
            LUU_HINH = False
            print("Không lưu được PNG (tạm tắt lưu hình):", str(e).splitlines()[0])

---
## 1. So sánh 6 chỉ số giữa 8 ngân hàng
### 1.1 Tổng quan: 6 chỉ số trong một hình
Mỗi hộp là phân phối 20 quý của 1 ngân hàng (CAR: 4 giá trị công bố). Hình này dùng cho báo cáo để người đọc thấy nhanh ngân hàng nào nổi bật ở chỉ số nào.

In [21]:
hien(ve_tong_quan_chi_so(b4), "p2_tong_quan_6_chi_so")

Không lưu được PNG (tạm tắt lưu hình): 


**Bảng số đi kèm** – trung bình cả giai đoạn 2021–2025 (CAR: trung bình 4 số công bố):

In [22]:
tb = b4.groupby("ma_nh", observed=True)[["roa", "roe", "nim", "npl", "ldr"]].mean()
tb["car"] = b4[b4["car_cong_bo"]].groupby("ma_nh", observed=True)["car"].mean()
tb = tb[cf.DS_CHI_SO].round(2)
tb.style.format("{:.2f}").highlight_max(color="#d4edda").highlight_min(color="#f8d7da")

,roa,roe,nim,npl,car,ldr
ma_nh,,,,,,
VCB,1.75,20.43,3.07,0.92,10.70,92.55
BID,0.93,17.88,2.60,1.50,9.02,104.00
CTG,1.12,17.82,2.88,1.30,9.24,105.19
TCB,2.84,17.63,4.64,0.92,14.98,118.65
MBB,2.46,23.26,4.87,1.47,11.02,107.43
ACB,2.19,23.07,3.84,1.08,12.08,102.94
STB,1.01,14.73,3.35,2.09,9.67,92.32
VPB,2.15,14.35,6.53,4.72,15.45,136.50


### 1.2 Diễn biến theo quý (line chart)
Đường xám đứt = trung bình 8 ngân hàng. CAR vẽ theo năm và chỉ dùng số công bố (2021–2024).

In [23]:
for cs in cf.DS_CHI_SO:
    hien(ve_duong_chi_so(b4, cs), f"p2_duong_{cs}")

### 1.3 Trung bình theo năm (bar chart)

In [24]:
for cs in cf.DS_CHI_SO:
    hien(ve_cot_trung_binh_nam(b4, cs), f"p2_cot_nam_{cs}")

Trung bình ngành theo năm (8 ngân hàng):

In [25]:
b4.groupby("nam")[["roa", "roe", "nim", "npl", "ldr"]].mean().round(2)

,roa,roe,nim,npl,ldr
nam,,,,,
2021,1.85,18.69,4.37,1.42,100.47
2022,2.02,20.53,4.43,1.51,107.23
2023,1.74,18.54,3.89,1.92,106.16
2024,1.77,18.37,3.78,1.98,109.38
2025,1.64,17.11,3.39,1.93,114.00


### 1.4 Phân phối theo ngân hàng (boxplot)
Hộp = 50% quý ở giữa, vạch giữa = trung vị, chấm = từng quý. Hộp càng dài thì chỉ số càng dao động.

In [26]:
for cs in cf.DS_CHI_SO:
    hien(ve_hop_chi_so(b4, cs), f"p2_hop_{cs}")

---
## 2. Xu hướng giá cổ phiếu
### 2.1 Giá điều chỉnh chuẩn hóa = 100 tại phiên đầu năm 2021
Chuẩn hóa để so 8 mã trên cùng trục dù giá tuyệt đối khác nhau. Giá trị 250 nghĩa là giá gấp 2,5 lần phiên 04/01/2021. Dùng giá **điều chỉnh** để loại ảnh hưởng của chia cổ tức, phát hành thêm.

In [27]:
hien(ve_gia_chuan_hoa(gia), "p2_gia_chuan_hoa")

In [28]:
# Giá trị chuẩn hóa ở phiên cuối năm của từng mã
gch = them_gia_chuan_hoa(gia)
gch["nam"] = gch["ngay"].dt.year
gch.groupby(["ma_nh", "nam"], observed=True)["gia_chuan_hoa"].last().unstack().round(0)

nam,2021,2022,2023,2024,2025
ma_nh,,,,,
VCB,103.0,104.0,124.0,141.0,133.0
BID,97.0,101.0,128.0,134.0,140.0
CTG,126.0,101.0,113.0,157.0,217.0
TCB,152.0,78.0,96.0,154.0,224.0
MBB,163.0,116.0,149.0,205.0,316.0
ACB,151.0,119.0,156.0,201.0,223.0
STB,179.0,128.0,159.0,210.0,330.0
VPB,188.0,141.0,159.0,167.0,257.0


### 2.2 Biên độ giá trong quý
Biên độ = (giá cao nhất − giá thấp nhất) / giá thấp nhất trong quý. Ô càng đậm, quý đó giá dao động càng mạnh.

In [29]:
hien(ve_bien_do_gia(b4), "p2_bien_do_gia")

In [30]:
print("Biên độ trung bình theo ngân hàng (%):")
print(b4.groupby("ma_nh", observed=True)["bien_dong_gia_max_min"].mean().round(1).sort_values().to_string())
print("\nBiên độ trung bình theo năm (%):")
print(b4.groupby("nam")["bien_dong_gia_max_min"].mean().round(1).to_string())

Biên độ trung bình theo ngân hàng (%):
ma_nh
VCB    19.8
ACB    21.3
BID    26.4
CTG    27.1
MBB    28.4
TCB    30.4
VPB    32.1
STB    33.7

Biên độ trung bình theo năm (%):
nam
2021    33.9
2022    36.9
2023    17.5
2024    19.9
2025    28.7


---
## 3. Sự kiện phi tài chính (hỗ trợ mục 5 của đề tài)
### 3.1 Số sự kiện theo ngân hàng và theo loại

In [31]:
hien(ve_dem_su_kien(su_kien, theo="ma_nh"), "p2_su_kien_theo_nh")
hien(ve_dem_su_kien(su_kien, theo="nam"), "p2_su_kien_theo_nam")

In [32]:
pd.crosstab(su_kien["ma_nh"], su_kien["loai_su_kien_nhan"], margins=True, margins_name="Tổng")

loai_su_kien_nhan,Chuyển đổi số - tiền số,Chính sách NHNN,Hợp tác chiến lược - trúng thầu,Khác,Tăng vốn,Đổi lãnh đạo,Tổng
ma_nh,,,,,,,
VCB,0,0,2,3,6,9,20
BID,0,1,2,0,9,7,19
CTG,0,0,0,0,8,7,15
TCB,1,1,5,0,11,1,19
MBB,1,0,0,2,10,4,17
ACB,1,0,1,0,6,1,9
STB,0,0,1,0,0,3,4
VPB,1,0,4,1,11,1,18
Tổng,4,2,15,6,61,33,121


### 3.2 Đánh dấu ngày công bố sự kiện trên đường giá
Tam giác = ngày có sự kiện (màu theo loại). Trong notebook/dashboard rê chuột vào tam giác để đọc tiêu đề tin. Phía dưới là khối lượng khớp lệnh. Hiện VCB làm ví dụ; PNG của cả 8 mã được lưu.

In [33]:
hien(ve_gia_su_kien(gia, su_kien, ma_nh="VCB"), "p2_gia_su_kien_VCB")
for nh in cf.DS_NGAN_HANG[1:]:
    hien(ve_gia_su_kien(gia, su_kien, ma_nh=nh), f"p2_gia_su_kien_{nh}", show=False)

---
## 4. Thử như trên dashboard
Cùng một hàm, đổi tham số lọc là ra biểu đồ khác – Hân chỉ cần nối các tham số này với ô chọn của Streamlit.

In [34]:
ve_duong_chi_so(b4, "nim", ma_nh=["VCB", "BID", "CTG", "TCB"], nam=[2024, 2025]).show()

In [35]:
ve_gia_chuan_hoa(gia, ma_nh=["VCB", "MBB", "STB"], nam=2025, dat_moc_lai=True).show()

In [36]:
n = len(list(cf.THU_MUC_HINH.glob("p2_*.png")))
print(f"Đã lưu {n} hình PNG vào {cf.THU_MUC_HINH}")

Đã lưu 0 hình PNG vào C:\Users\hoang\OneDrive\Nhóm5-PTDL\outputs\figures


---
## 5. Điều rút ra từ Phần 2 (mô tả, chưa kiểm định)

**Sinh lời (gợi ý cho câu 1).** Tính trung bình cả giai đoạn, ROE cao nhất là MBB (23,3%) và ACB (23,1%), tiếp theo VCB (20,4%); thấp nhất là VPB (14,4%) và STB (14,7%). ROA cao nhất lại là TCB (2,8%) vì TCB có vốn dày (CAR ~15%), nên ROE không nổi bật bằng ROA. Mặt bằng ROE ngành đạt đỉnh năm 2022 (20,5%) rồi giảm dần còn 17,1% năm 2025.

**Biên lãi, nợ xấu, an toàn vốn (gợi ý cho câu 2).** NIM của ba ngân hàng quốc doanh thấp nhất nhóm (BID 2,6%, CTG 2,9%, VCB 3,1%), VPB cao nhất (6,5%). Trung bình giai đoạn: NIM quốc doanh 2,85% so với tư nhân 4,64%; NPL quốc doanh 1,24% so với tư nhân 2,06% – nhưng NPL tư nhân bị VPB (4,7%) kéo lên. NIM toàn ngành giảm từ 4,4% (2021) xuống 3,4% (2025), NPL tăng từ 1,4% lên khoảng 1,9% từ 2023. CAR của BID, CTG thấp nhất (~9,0–9,2%; 3/4 năm là số nhóm tra thủ công).

**Giá cổ phiếu (gợi ý cho câu 3).** Cả 8 mã đều tăng so với đầu 2021, nhưng chênh lệch lớn: STB (330) và MBB (316) tăng hơn 3 lần, còn VCB (133) và BID (140) tăng ít nhất. Giá giảm mạnh cùng lúc vào nửa cuối 2022 rồi phục hồi. Biên độ giá cao năm 2021–2022 (34–37%/quý), thấp năm 2023–2024 (18–20%), tăng lại năm 2025 (29%). VCB dao động ít nhất (20%/quý), STB nhiều nhất (34%); VPB Q3/2025 biên độ tới 112%. Đáng chú ý: STB có ROE thấp nhưng giá tăng mạnh nhất, VCB có ROE cao nhưng giá tăng ít nhất – dấu hiệu quan hệ giữa chỉ số tài chính và giá có thể không đơn giản; câu 3 sẽ kiểm tra bằng tương quan.

**Sự kiện.** 121 sự kiện, trong đó tăng vốn chiếm 61 (50%), đổi lãnh đạo 33. Số sự kiện chênh nhau nhiều giữa các ngân hàng: VCB 20, BID và TCB 19, nhưng ACB chỉ 9 và STB chỉ 4 – có thể do độ phủ của nguồn tin chứ không hẳn do ngân hàng ít sự kiện. Đây là hạn chế cần báo Quân khi dùng biến sự kiện cho mô hình.

**Lưu ý khi diễn giải.** Đây là số trung bình và hình ảnh, chưa phải kết luận thống kê. Các khác biệt nêu trên sẽ được kiểm tra bằng xếp hạng (Phần 3), kiểm định (Phần 4) và tương quan (Phần 5).